# AirScope: Air Quality Telemetry Preprocessing & Big Data Pipeline
### Course: Big Data Technologies & Data Analytics
**Dataset**: `india_air_quality_2024_hourly.csv` (88,340 Hourly Records across 10 Cities)
**Objective**: Perform end-to-end data extraction, anomaly cleaning, null imputation, CPCB NAQI calculation, and preparation for Redis Cloud in-memory storage.

## 1. Import Libraries & Load Dataset

In [ ]:
import pandas as pd
import numpy as np
import json
from datetime import datetime

# Load the 2024 hourly dataset
df = pd.read_csv('india_air_quality_2024_hourly.csv')

print(f"Total rows: {df.shape[0]:,}")
print(f"Total columns: {df.shape[1]}")
df.head()

## 2. Dataset Profiling & City Distribution

In [ ]:
# Inspect observation distribution across Indian cities
city_counts = df['city'].value_counts()
print("Observations per Metropolitan Hub:")
print(city_counts)

# Check data types and non-null values
df.info()

## 3. Data Cleaning & Preprocessing
### Key Steps:
1. **Drop Invalid / Negative Values**: Concentrations cannot be physically negative ($< 0$).
2. **Null Imputation Strategy**: Maintain missing values as `NaN`/`null` rather than filling with 0, preventing downward bias in averages.
3. **Timestamp Normalization**: Convert `datetime` into standard ISO and extract epoch integers.

In [ ]:
# Numeric pollutants list
pollutants = ['pm25', 'pm10', 'no2', 'so2', 'co', 'o3', 'temperature', 'humidity', 'wind_speed']

# Replace any sensor negative artifacts with NaN
for col in ['pm25', 'pm10', 'no2', 'so2', 'co', 'o3']:
    neg_count = (df[col] < 0).sum()
    if neg_count > 0:
        print(f"Removing {neg_count} negative sensor artifacts from {col}")
        df.loc[df[col] < 0, col] = np.nan

# Convert datetime column to pandas datetime object
df['datetime'] = pd.to_datetime(df['datetime'])
df['date'] = df['datetime'].dt.strftime('%Y-%m-%d')
df['time'] = df['datetime'].dt.strftime('%H:%M:%S')
df['epoch'] = df['datetime'].astype('int64') // 10**9

# Summary statistics after cleaning
df[pollutants].describe().round(2)

## 4. CPCB National AQI (NAQI) Mathematical Formulation
Implementing the linear interpolation equation defined by the Central Pollution Control Board (CPCB) for PM2.5 breakpoints:

$$I_p = I_{\text{low}} + \frac{I_{\text{high}} - I_{\text{low}}}{B_{\text{high}} - B_{\text{low}}} \times (C_p - B_{\text{low}})$$

In [ ]:
def calculate_cpcb_aqi(pm25):
    if pd.isna(pm25):
        return np.nan
    if pm25 <= 30:
        return round((pm25 * 50) / 30)
    elif pm25 <= 60:
        return round(50 + ((pm25 - 30) * 50) / 30)
    elif pm25 <= 90:
        return round(100 + ((pm25 - 60) * 100) / 30)
    elif pm25 <= 120:
        return round(200 + ((pm25 - 90) * 100) / 30)
    elif pm25 <= 250:
        return round(300 + ((pm25 - 120) * 100) / 130)
    else:
        return round(400 + ((pm25 - 250) * 100) / 130)

def get_aqi_category(aqi):
    if pd.isna(aqi): return "Unknown"
    if aqi <= 50: return "Good"
    if aqi <= 100: return "Satisfactory"
    if aqi <= 200: return "Moderate"
    if aqi <= 300: return "Poor"
    if aqi <= 400: return "Very Poor"
    return "Severe"

# Apply AQI calculation on PM2.5 concentrations
df['aqi'] = df['pm25'].apply(calculate_cpcb_aqi)
df['category'] = df['aqi'].apply(get_aqi_category)

print("AQI Category Distribution across all 88,340 hours:")
print(df['category'].value_counts())

## 5. Temporal Aggregation & Hierarchical Bucketing
To optimize memory storage in Redis Cloud and achieve sub-millisecond retrieval:
- Group 24 hourly readings for each city and calendar day.
- Calculate daily averages for PM2.5, PM10, NO2, SO2, CO, and AQI.
- This reduces total database keys from **88,340** to **3,660**.

In [ ]:
# Compute Daily Averages per City
daily_df = df.groupby(['city', 'date']).agg({
    'pm25': 'mean',
    'pm10': 'mean',
    'no2': 'mean',
    'so2': 'mean',
    'co': 'mean',
    'o3': 'mean',
    'temperature': 'mean',
    'humidity': 'mean',
    'wind_speed': 'mean',
    'aqi': 'mean'
}).round(1).reset_index()

print(f"Aggregated into {len(daily_df):,} City-Day Buckets")
daily_df.head(10)

## 6. City-Level Comparative Analytics (Annual 2024)

In [ ]:
# Annual City Ranking by Average AQI
city_summary = df.groupby('city').agg({
    'aqi': 'mean',
    'pm25': 'mean',
    'pm10': 'mean',
    'no2': 'mean'
}).round(1).sort_values(by='aqi', ascending=False)

city_summary['Status'] = city_summary['aqi'].apply(get_aqi_category)
city_summary

## 7. Big Data Export: Preparing JSON Payloads for Redis Ingestion

In [ ]:
# Sample of how a day bucket is structured for Redis SET command
sample_city = 'Delhi'
sample_date = '2024-01-01'

sample_hours = df[(df['city'] == sample_city) & (df['date'] == sample_date)]
payload = {
    "date": sample_date,
    "city": sample_city,
    "avg_pm25": float(sample_hours['pm25'].mean().round(1)),
    "avg_pm10": float(sample_hours['pm10'].mean().round(1)),
    "aqi": int(round(sample_hours['aqi'].mean())),
    "hourly_count": len(sample_hours),
    "hourly_readings": sample_hours[['time', 'pm25', 'pm10', 'no2', 'aqi']].to_dict(orient='records')
}

print(f"Sample Redis Key: aqi:daily:{sample_city.lower()}:{sample_date}")
print(json.dumps(payload, indent=2)[:500] + "\n... [Truncated for preview]")